<a href="https://colab.research.google.com/github/ahmedaminkhaled/MobileNet-V4--pytorch/blob/main/implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as Ff


In [6]:
class ConvBNAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        groups=1,
        activation=True
    ):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=groups,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        ]

        if activation:
            layers.append(nn.ReLU(inplace=True))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [7]:
class UIB(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        start_dw_kernel,
        middle_dw_kernel,
        stride,
        expand_ratio
    ):
        super().__init__()

        original_stride = stride
        hidden_dim = int(in_channels * expand_ratio)

        layers = []

        if start_dw_kernel > 0:
            layers.append(
                ConvBNAct(
                    in_channels,
                    in_channels,
                    start_dw_kernel,
                    stride=stride,
                    groups=in_channels
                )
            )
            stride = 1

        layers.append(
            ConvBNAct(
                in_channels,
                hidden_dim,
                1
            )
        )

        if middle_dw_kernel > 0:
            layers.append(
                ConvBNAct(
                    hidden_dim,
                    hidden_dim,
                    middle_dw_kernel,
                    stride=stride,
                    groups=hidden_dim
                )
            )

        layers.append(
            ConvBNAct(
                hidden_dim,
                out_channels,
                1,
                activation=False
            )
        )

        self.block = nn.Sequential(*layers)

        self.use_residual = (
            original_stride == 1
            and in_channels == out_channels
        )

    def forward(self, x):
        y = self.block(x)

        if self.use_residual:
            y = y + x

        return y

In [8]:
stage1 = nn.Sequential(

    UIB(64, 96, 5, 5, 2, 3),

    UIB(96, 96, 0, 3, 1, 2),
    UIB(96, 96, 0, 3, 1, 2),
    UIB(96, 96, 0, 3, 1, 2),
    UIB(96, 96, 0, 3, 1, 2),

    UIB(96, 96, 3, 0, 1, 4)
)

In [9]:
stage2 = nn.Sequential(

    UIB(96, 128, 3, 3, 2, 6),

    UIB(128, 128, 5, 5, 1, 4),

    UIB(128, 128, 0, 5, 1, 4),

    UIB(128, 128, 0, 5, 1, 3),

    UIB(128, 128, 0, 3, 1, 4),

    UIB(128, 128, 0, 3, 1, 4)
)

In [10]:
class MobileNetV4Custom(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()

        self.initial = ConvBNAct(3, 32, 3, stride=2)

        self.lightweight = nn.Sequential(
            ConvBNAct(32, 32, 3, stride=2),
            ConvBNAct(32, 32, 1)
        )

        self.middle = nn.Sequential(
            ConvBNAct(32, 96, 3, stride=2),
            ConvBNAct(96, 64, 1)
        )

        self.stage1 = stage1
        self.stage2 = stage2

        self.tail = nn.Sequential(
            ConvBNAct(128, 960, 1),
            ConvBNAct(960, 1280, 1)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(
            1280,
            num_classes
        )

    def forward(self, x):

        x = self.initial(x)

        x = self.lightweight(x)

        x = self.middle(x)

        x = self.stage1(x)

        x = self.stage2(x)

        x = self.tail(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x

In [12]:
model = MobileNetV4Custom(num_classes=1000)

x = torch.randn(1, 3, 224, 224)

y = model(x)

print(y.shape)
print(y)

torch.Size([1, 1000])
tensor([[-9.6492e-02, -5.7133e-02,  1.6321e-01,  1.0286e-01,  1.9709e-01,
          1.7560e-01, -6.2426e-02,  2.9635e-01,  5.3423e-02, -4.6822e-02,
         -3.2920e-01, -2.5274e-01, -5.3083e-01,  1.5433e-01,  2.9709e-01,
          8.1148e-02,  8.3708e-02, -1.2245e-01, -1.6756e-01, -1.3347e-01,
         -1.8321e-01,  2.4360e-01, -9.1369e-02, -1.0977e-02, -1.5935e-01,
          1.6046e-02, -2.4657e-01, -1.7430e-01,  1.6835e-01, -2.7240e-01,
         -1.2651e-01,  1.0292e-01, -1.2895e-01,  4.4859e-01,  3.0849e-01,
          2.2238e-01, -5.5810e-01,  1.7978e-01,  8.2925e-02, -2.0669e-01,
         -1.4429e-01, -6.5317e-02,  1.8108e-01, -2.1144e-01, -1.7972e-01,
          3.0103e-01, -1.7512e-02,  2.6994e-01,  1.7592e-01, -2.4782e-01,
          6.5482e-01,  2.8274e-01, -1.1585e-01,  4.1206e-01, -2.9380e-02,
         -4.3618e-02, -1.6117e-01,  9.8822e-02,  1.9024e-01,  1.3576e-01,
         -2.8949e-02,  4.1384e-01,  1.1525e-01, -2.5882e-02, -3.3560e-01,
          5.2492